<a href="https://colab.research.google.com/github/NathanAndrewsBYU/Stylometry-Federalist-Papers/blob/main/notebooks/burrows_delta_disputed_included.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Burrows' Delta: Balanced, Disputed-Included Experiment

Runs the classic Burrows' Delta function-word baseline on the Federalist
Papers, for comparison against the GPT-2 perplexity method from the main
project (run on BYU's cluster).

**What this notebook does:**
1. Downloads the Federalist Papers from Project Gutenberg and sorts them into
   author folders (Hamilton, Madison, disputed, etc.) — no cluster needed,
   Colab has normal internet access.
2. Folds the 12 disputed papers into Madison's corpus (14 known + 12 disputed
   = 26 total), auto-balances Hamilton down to 26 to match (correcting the
   51-vs-14 corpus-size imbalance that biases Delta toward Hamilton).
3. Runs balanced leave-one-out validation across multiple balance seeds.
4. Reports accuracy overall and specifically on the 12 disputed papers.

No GPU, no fine-tuning — this runs in seconds.

## 1. Download and prepare the corpus

In [ ]:
import os
import re
import glob
import requests

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/1404/pg1404.txt"
CORPUS_ROOT = "corpus/federalist"

HAMILTON_SOLO = set([1] + list(range(6, 10)) + list(range(11, 14)) + list(range(15, 18)) +
                     list(range(21, 37)) + list(range(59, 62)) + list(range(65, 86)))
MADISON_SOLO = set([10, 14] + list(range(37, 49)))
JAY = set([2, 3, 4, 5, 64])
JOINT = set([18, 19, 20])
DISPUTED = set(list(range(49, 59)) + [62, 63])


def folder_for(number):
    if number in HAMILTON_SOLO:
        return "hamilton"
    if number in MADISON_SOLO:
        return "madison"
    if number in JAY:
        return "jay"
    if number in JOINT:
        return "joint"
    if number in DISPUTED:
        return "disputed"
    return None


def prepare_corpus():
    print("Downloading Federalist Papers from Project Gutenberg...")
    response = requests.get(GUTENBERG_URL, timeout=30)
    response.raise_for_status()
    raw_text = response.text
    print(f"Downloaded {len(raw_text):,} characters.")

    start_marker = "*** START OF"
    end_marker = "*** END OF"
    start_idx = raw_text.find(start_marker)
    end_idx = raw_text.find(end_marker)
    if start_idx != -1 and end_idx != -1:
        body_start = raw_text.find("\n", start_idx) + 1
        body_text = raw_text[body_start:end_idx]
    else:
        print("WARNING: could not find Gutenberg markers, using full text.")
        body_text = raw_text

    header_pattern = re.compile(r"FEDERALIST\.?\s+No\.?\s+(\d+)", re.IGNORECASE)
    matches = list(header_pattern.finditer(body_text))
    print(f"Found {len(matches)} header matches (expect at least 85).")

    papers = {}
    for i, match in enumerate(matches):
        number = int(match.group(1))
        section_start = match.end()
        section_end = matches[i + 1].start() if i + 1 < len(matches) else len(body_text)
        text = body_text[section_start:section_end].strip()
        if number not in papers or len(text) > len(papers[number]):
            papers[number] = text

    missing = sorted(set(range(1, 86)) - set(papers.keys()))
    if missing:
        print(f"WARNING: missing papers {missing}")

    assert len(HAMILTON_SOLO) == 51 and len(MADISON_SOLO) == 14
    assert len(JAY) == 5 and len(JOINT) == 3 and len(DISPUTED) == 12

    for folder in ["hamilton", "madison", "jay", "joint", "disputed"]:
        os.makedirs(os.path.join(CORPUS_ROOT, folder), exist_ok=True)

    written = 0
    for number, text in papers.items():
        folder = folder_for(number)
        if folder is None:
            continue
        out_path = os.path.join(CORPUS_ROOT, folder, f"federalist_{number:02d}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)
        written += 1

    print(f"\nWrote {written} paper files into {CORPUS_ROOT}/")
    for folder in ["hamilton", "madison", "jay", "joint", "disputed"]:
        n = len(glob.glob(os.path.join(CORPUS_ROOT, folder, "*.txt")))
        print(f"  {folder}: {n} files")


prepare_corpus()

Downloaded 1,186,743 characters.
Found 85 header matches (expect at least 85).

Wrote 85 paper files into corpus/federalist/
  hamilton: 51 files
  madison: 14 files
  jay: 5 files
  joint: 3 files
  disputed: 12 files


## 2. Burrows' Delta implementation (balanced, disputed-included)

In [ ]:
import csv
import random
import statistics
from collections import Counter

N_FEATURES = 100


def list_txt_files(subfolder):
    return sorted(glob.glob(os.path.join(CORPUS_ROOT, subfolder, "*.txt")))


def clean_text(raw):
    text = raw
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"PUBLIUS\.?\s*$", "", text.strip())
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text):
    return re.findall(r"[a-z\']+", text.lower())


def load_tokens(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return tokenize(clean_text(f.read()))


def word_frequencies(tokens, vocabulary):
    total = len(tokens)
    counts = Counter(tokens)
    return {w: (counts.get(w, 0) / total) * 1000 for w in vocabulary}


def build_vocabulary(all_files, n_features):
    counts = Counter()
    for path in all_files:
        counts.update(load_tokens(path))
    return [w for w, _ in counts.most_common(n_features)]


def author_profile(files, vocabulary):
    freq_rows = [word_frequencies(load_tokens(f), vocabulary) for f in files]
    profile = {}
    for w in vocabulary:
        values = [row[w] for row in freq_rows]
        mean = statistics.mean(values)
        stdev = statistics.stdev(values) if len(values) > 1 else 0
        stdev = stdev if stdev > 0 else 1e-6
        profile[w] = (mean, stdev)
    return profile


def burrows_delta(paper_freqs, profile, vocabulary):
    z_scores = [abs((paper_freqs[w] - profile[w][0]) / profile[w][1]) for w in vocabulary]
    return sum(z_scores) / len(z_scores)


def balance_hamilton(hamilton_files, madison_files, balance_seed):
    """Subsample Hamilton\'s papers down to Madison\'s count, reproducibly."""
    if balance_seed is None or len(hamilton_files) <= len(madison_files):
        return hamilton_files
    rng = random.Random(balance_seed)
    return sorted(rng.sample(hamilton_files, len(madison_files)))


def leave_one_out_delta(hamilton_files, madison_files, n_features=N_FEATURES,
                         balance_seed=42, verbose=True):
    balanced_hamilton = balance_hamilton(hamilton_files, madison_files, balance_seed)

    if verbose and balance_seed is not None and len(balanced_hamilton) < len(hamilton_files):
        print(f"Balancing: using {len(balanced_hamilton)}/{len(hamilton_files)} Hamilton "
              f"papers (seed={balance_seed}) vs. {len(madison_files)} Madison papers.")

    all_files = balanced_hamilton + madison_files
    vocabulary = build_vocabulary(all_files, n_features)
    if verbose:
        print(f"Using top {len(vocabulary)} most frequent words as features.")

    all_papers = [("hamilton", f) for f in balanced_hamilton] + [("madison", f) for f in madison_files]
    results = []

    for true_author, held_out_file in all_papers:
        remaining_hamilton = [f for f in balanced_hamilton if f != held_out_file]
        remaining_madison = [f for f in madison_files if f != held_out_file]

        ham_profile = author_profile(remaining_hamilton, vocabulary)
        mad_profile = author_profile(remaining_madison, vocabulary)

        held_out_freqs = word_frequencies(load_tokens(held_out_file), vocabulary)
        delta_h = burrows_delta(held_out_freqs, ham_profile, vocabulary)
        delta_m = burrows_delta(held_out_freqs, mad_profile, vocabulary)

        predicted_author = "hamilton" if delta_h < delta_m else "madison"
        correct = predicted_author == true_author

        results.append({
            "file": os.path.basename(held_out_file),
            "true_author": true_author,
            "predicted_author": predicted_author,
            "delta_hamilton": delta_h,
            "delta_madison": delta_m,
            "correct": correct,
            "balance_seed": balance_seed,
        })

    accuracy = sum(r["correct"] for r in results) / len(results)
    if verbose:
        print(f"Overall accuracy (balance_seed={balance_seed}): {accuracy:.1%} "
              f"({sum(r['correct'] for r in results)}/{len(results)})")
    return results


def save_results_csv(results, out_path):
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"Saved {len(results)} results to {out_path}")

## 3. Run the disputed-included balanced experiment

Folds the 12 disputed papers into Madison's corpus (26 total), auto-balances
Hamilton to match, and sweeps several balance seeds so the result doesn't
depend on which 26 of Hamilton's 51 papers happen to get picked.

In [ ]:
BALANCE_SEEDS = [1, 2, 3, 4, 5]

hamilton_files = list_txt_files("hamilton")
madison_files = list_txt_files("madison")
disputed_files = list_txt_files("disputed")
print(f"Loaded {len(hamilton_files)} Hamilton, {len(madison_files)} Madison, "
      f"{len(disputed_files)} disputed papers.")

expanded_madison = madison_files + disputed_files
disputed_basenames = {os.path.basename(f) for f in disputed_files}
print(f"Expanded Madison corpus: {len(madison_files)} known + {len(disputed_files)} "
      f"disputed = {len(expanded_madison)} total.\n")

all_seed_results = {}
combined_results = []
for seed in BALANCE_SEEDS:
    print(f"=== balance_seed={seed} ===")
    results = leave_one_out_delta(hamilton_files, expanded_madison, balance_seed=seed)
    for r in results:
        r["was_disputed"] = r["file"] in disputed_basenames
    all_seed_results[seed] = results
    combined_results.extend(results)
    print()

save_results_csv(combined_results, "results/traditional_burrows_delta_disputed_included.csv")

Loaded 51 Hamilton, 14 Madison, 12 disputed papers.
Expanded Madison corpus: 14 known + 12 disputed = 26 total.

=== balance_seed=1 ===
Balancing: using 26/51 Hamilton papers (seed=1) vs. 26 Madison papers.
Using top 100 most frequent words as features.
Overall accuracy (balance_seed=1): 96.2% (50/52)

=== balance_seed=2 ===
Balancing: using 26/51 Hamilton papers (seed=2) vs. 26 Madison papers.
Using top 100 most frequent words as features.
Overall accuracy (balance_seed=2): 96.2% (50/52)

=== balance_seed=3 ===
Balancing: using 26/51 Hamilton papers (seed=3) vs. 26 Madison papers.
Using top 100 most frequent words as features.
Overall accuracy (balance_seed=3): 94.2% (49/52)

=== balance_seed=4 ===
Balancing: using 26/51 Hamilton papers (seed=4) vs. 26 Madison papers.
Using top 100 most frequent words as features.
Overall accuracy (balance_seed=4): 100.0% (52/52)

=== balance_seed=5 ===
Balancing: using 26/51 Hamilton papers (seed=5) vs. 26 Madison papers.
Using top 100 most frequent 

## 4. Summarize results

In [ ]:
import pandas as pd

summary_rows = []
for seed, results in all_seed_results.items():
    n_disputed = sum(1 for r in results if r["was_disputed"])
    correct_disputed = sum(1 for r in results if r["was_disputed"] and r["correct"])
    overall_accuracy = sum(r["correct"] for r in results) / len(results)
    disputed_accuracy = correct_disputed / n_disputed if n_disputed else float("nan")
    summary_rows.append({
        "balance_seed": seed,
        "overall_accuracy": overall_accuracy,
        "disputed_accuracy": disputed_accuracy,
        "n_disputed": n_disputed,
        "n_total": len(results),
    })

summary_df = pd.DataFrame(summary_rows)
save_results_csv(summary_rows, "results/traditional_burrows_delta_disputed_included_summary.csv")
summary_df

Saved 5 results to results/traditional_burrows_delta_disputed_included_summary.csv


,balance_seed,overall_accuracy,disputed_accuracy,n_disputed,n_total
0,1,0.961538,1.0,12,52
1,2,0.961538,1.0,12,52
2,3,0.942308,1.0,12,52
3,4,1.000000,1.0,12,52
4,5,0.961538,1.0,12,52


## 5. Per-paper breakdown for the 12 disputed papers

Which disputed papers get misclassified, and how often, across seeds.

In [ ]:
df = pd.DataFrame(combined_results)
disputed_df = df[df["was_disputed"]]
pivot = disputed_df.pivot_table(index="file", columns="balance_seed", values="correct")
pivot["times_correct"] = pivot.sum(axis=1)
pivot.sort_values("times_correct")

balance_seed,1,2,3,4,5,times_correct
file,,,,,,
federalist_49.txt,1.0,1.0,1.0,1.0,1.0,5.0
federalist_50.txt,1.0,1.0,1.0,1.0,1.0,5.0
federalist_51.txt,1.0,1.0,1.0,1.0,1.0,5.0
federalist_52.txt,1.0,1.0,1.0,1.0,1.0,5.0
federalist_53.txt,1.0,1.0,1.0,1.0,1.0,5.0
federalist_54.txt,1.0,1.0,1.0,1.0,1.0,5.0
federalist_55.txt,1.0,1.0,1.0,1.0,1.0,5.0
federalist_56.txt,1.0,1.0,1.0,1.0,1.0,5.0
federalist_57.txt,1.0,1.0,1.0,1.0,1.0,5.0


## 6. Download results

In [ ]:
from google.colab import files

files.download("results/traditional_burrows_delta_disputed_included.csv")
files.download("results/traditional_burrows_delta_disputed_included_summary.csv")

FileNotFoundError: Cannot find file: results/traditional_burrows_delta_disputed_included.csv